In [1]:
import json

In [2]:
launcher_arg = '{"training_folder":"/tmp","params":{"preparation_footpaths":{"preparation_footpaths_max_length":3000,"preparation_footpaths_speed":3,"preparation_footpaths_n_clusters":3000},"preparation_ntlegs":{"preparation_ntlegs_n_ntlegs":5,"preparation_ntlegs_max_ntleg_length":5000,"preparation_ntlegs_long_leg_speed":7,"preparation_ntlegs_short_leg_speed":3,"preparation_ntlegs_threshold":500},"preparation_logit":{"preparation_logit_time":-5.5E-4,"preparation_logit_price":-1,"preparation_logit_transfers":-0.25,"preparation_logit_mode":1,"preparation_logit_pt_mode":0.5,"preparation_logit_pt_path":0.1},"car_owner":{"car_owner_walk":0,"car_owner_bus":12,"car_owner_subway":1,"car_owner_rail":4.5,"car_owner_car":10},"pt_captive":{"pt_captive_walk":0,"pt_captive_bus":1.5,"pt_captive_subway":1.5,"pt_captive_rail":0.5,"pt_captive_car":0},"analysis_mode_utility":{"analysis_mode_utility_how":"max"}}}'

In [3]:
params = json.loads(launcher_arg)

In [4]:
params['params']['car_owner']['car_owner_subway']

1

In [5]:
json.loads(json.dumps(launcher_arg))

'{"training_folder":"/tmp","params":{"preparation_footpaths":{"preparation_footpaths_max_length":3000,"preparation_footpaths_speed":3,"preparation_footpaths_n_clusters":3000},"preparation_ntlegs":{"preparation_ntlegs_n_ntlegs":5,"preparation_ntlegs_max_ntleg_length":5000,"preparation_ntlegs_long_leg_speed":7,"preparation_ntlegs_short_leg_speed":3,"preparation_ntlegs_threshold":500},"preparation_logit":{"preparation_logit_time":-5.5E-4,"preparation_logit_price":-1,"preparation_logit_transfers":-0.25,"preparation_logit_mode":1,"preparation_logit_pt_mode":0.5,"preparation_logit_pt_path":0.1},"car_owner":{"car_owner_walk":0,"car_owner_bus":12,"car_owner_subway":1,"car_owner_rail":4.5,"car_owner_car":10},"pt_captive":{"pt_captive_walk":0,"pt_captive_bus":1.5,"pt_captive_subway":1.5,"pt_captive_rail":0.5,"pt_captive_car":0},"analysis_mode_utility":{"analysis_mode_utility_how":"max"}}}'

In [6]:
json.loads('{}')

{}

In [ ]:
import boto3

ecs = boto3.client('ecs', region_name='ca-central-1')


In [79]:
function_name = 'quetzal-test'
scenario_path_S3 = 'spark_example'
cluster = f'arn:aws:ecs:ca-central-1:142023388927:cluster/{function_name}'
task_definition = f'arn:aws:ecs:ca-central-1:142023388927:task-definition/{function_name}-task'
response = ecs.run_task(
	cluster=cluster,
	launchType='FARGATE',
	taskDefinition=task_definition,
	count=1,
	networkConfiguration={
		'awsvpcConfiguration': {
			'subnets': ['subnet-04ff36f2b80327321'],
			'securityGroups': ['sg-087728e78f1a2c4c2'],
			'assignPublicIp': 'ENABLED',
		}
	},
	overrides={
		'containerOverrides': [
			{
				'name': function_name,
				'environment': [
					{'name': 'notebook_path', 'value': 'scripts/main.py'},
					{'name': 'scenario_path_S3', 'value': scenario_path_S3},
					{'name': 'launcher_arg', 'value': '{"foo": "bar"}'},
					{'name': 'metadata', 'value': '{"user": "simon"}'},
				],
			}
		]
	},
)

# task_arn = response['tasks'][0]['taskArn']

In [80]:
task_arn = response['tasks'][0]['taskArn']
task_arn

'arn:aws:ecs:ca-central-1:142023388927:task/quetzal-test/31610a8043fe41de854e512000433606'

In [81]:
# print(ecs.describe_task_definition(
#         taskDefinition="quetzal-test-task"
#     )["taskDefinition"]["taskDefinitionArn"]
# )

In [97]:
task_arn

'arn:aws:ecs:ca-central-1:142023388927:task/quetzal-test/31610a8043fe41de854e512000433606'

In [ ]:
response = ecs.describe_tasks(cluster=cluster, tasks=[task_arn])

In [93]:
response['tasks'][0]['lastStatus']

'DEPROVISIONING'

In [94]:
response['tasks'][0]['desiredStatus']

'STOPPED'

In [96]:
response['tasks'][0]

{'attachments': [{'id': '8e8e4533-48b1-4c43-b38e-e4867e0a257a',
   'type': 'ElasticNetworkInterface',
   'status': 'DETACHED',
   'details': [{'name': 'subnetId', 'value': 'subnet-04ff36f2b80327321'},
    {'name': 'networkInterfaceId', 'value': 'eni-0dc0812d9d9f9bc13'},
    {'name': 'macAddress', 'value': '02:21:4f:19:bb:d1'},
    {'name': 'privateDnsName',
     'value': 'ip-10-0-101-161.ca-central-1.compute.internal'},
    {'name': 'privateIPv4Address', 'value': '10.0.101.161'}]}],
 'attributes': [{'name': 'ecs.cpu-architecture', 'value': 'x86_64'}],
 'availabilityZone': 'ca-central-1a',
 'clusterArn': 'arn:aws:ecs:ca-central-1:142023388927:cluster/quetzal-test',
 'containers': [{'containerArn': 'arn:aws:ecs:ca-central-1:142023388927:container/quetzal-test/31610a8043fe41de854e512000433606/a4376c35-70bd-4b35-9de0-b22ac2217677',
   'taskArn': 'arn:aws:ecs:ca-central-1:142023388927:task/quetzal-test/31610a8043fe41de854e512000433606',
   'name': 'quetzal-test',
   'image': '142023388927.d

In [85]:
# waiter = ecs.get_waiter("tasks_stopped")

# waiter.wait(
#     cluster="quetzal-test",
#     tasks=[task_arn]
# )

In [ ]:
try:
	response = ecs.stop_task(cluster=cluster, task=task_arn, reason='User cancelled job')
except:
	print('ypo')

ypo


In [44]:
response['task']

{'attachments': [{'id': '0edcb0d7-b2dc-4444-9896-6259835fe380',
   'type': 'ElasticNetworkInterface',
   'status': 'ATTACHING',
   'details': [{'name': 'subnetId', 'value': 'subnet-04ff36f2b80327321'},
    {'name': 'networkInterfaceId', 'value': 'eni-0532c74d52ae86fe1'},
    {'name': 'macAddress', 'value': '02:6c:aa:ff:a2:3d'},
    {'name': 'privateDnsName',
     'value': 'ip-10-0-101-197.ca-central-1.compute.internal'},
    {'name': 'privateIPv4Address', 'value': '10.0.101.197'}]}],
 'attributes': [{'name': 'ecs.cpu-architecture', 'value': 'x86_64'}],
 'availabilityZone': 'ca-central-1a',
 'clusterArn': 'arn:aws:ecs:ca-central-1:142023388927:cluster/quetzal-test',
 'containers': [{'containerArn': 'arn:aws:ecs:ca-central-1:142023388927:container/quetzal-test/ab089bca172d4d8dbc71d61176652ee4/ab75d916-dac4-4508-8c9a-8fb40bb29403',
   'taskArn': 'arn:aws:ecs:ca-central-1:142023388927:task/quetzal-test/ab089bca172d4d8dbc71d61176652ee4',
   'name': 'quetzal-test',
   'image': '142023388927.

In [ ]:
response = ecs.list_tasks(cluster=cluster, desiredStatus='RUNNING')

In [147]:
response['taskArns']

[]

In [ ]:
response = ecs.describe_tasks(
	cluster=cluster, tasks=['arn:aws:ecs:ca-central-1:142023388927:task/quetzal-test/5130d8d0420b4f6c9c169ec079fb7da4']
)

In [158]:
response['tasks'][0]['lastStatus']

'STOPPED'

In [ ]:
response['tasks'][0]['stoppedReason']

'Essential container in task exited'

In [187]:
response['tasks'][0]['overrides']['containerOverrides'][0]['environment']

[{'name': 'steps', 'value': '[]'},
 {'name': 'scenario_path_S3', 'value': 'api_test/'},
 {'name': 'launcher_arg',
  'value': "{'training_folder': '/tmp', 'params': {'preparation_footpaths': {'preparation_footpaths_max_length': 3000, 'preparation_footpaths_speed': 3, 'preparation_footpaths_n_clusters': 3000}, 'preparation_ntlegs': {'preparation_ntlegs_n_ntlegs': 5, 'preparation_ntlegs_max_ntleg_length': 5000, 'preparation_ntlegs_long_leg_speed': 7, 'preparation_ntlegs_short_leg_speed': 3, 'preparation_ntlegs_threshold': 500}, 'preparation_logit': {'preparation_logit_time': -0.00055, 'preparation_logit_price': -1, 'preparation_logit_transfers': -0.25, 'preparation_logit_mode': 1, 'preparation_logit_pt_mode': 0.5, 'preparation_logit_pt_path': 0.1}, 'car_owner': {'car_owner_walk': 0, 'car_owner_bus': 12, 'car_owner_subway': 1, 'car_owner_rail': 4.5, 'car_owner_car': 10}, 'pt_captive': {'pt_captive_walk': 0, 'pt_captive_bus': 1.5, 'pt_captive_subway': 1.5, 'pt_captive_rail': 0.5, 'pt_captiv

In [180]:
scenario = 'r'
for task in response['tasks']:
	envs = task['overrides']['containerOverrides'][0]['environment']
	filtered_envs = [v for v in envs if v['name'] == 'scenario_path_S3']
	if len(filtered_envs) > 0:
		scen = filtered_envs[0]['value'].strip('/')
		if scen == scenario:
			print('tes')

In [186]:
task['taskArn']

'arn:aws:ecs:ca-central-1:142023388927:task/quetzal-test/5130d8d0420b4f6c9c169ec079fb7da4'

In [188]:
response['tasks'][0]

{'attachments': [{'id': 'bafbbc17-413e-497d-8746-b662e6003d99',
   'type': 'ElasticNetworkInterface',
   'status': 'DELETED',
   'details': [{'name': 'subnetId', 'value': 'subnet-04ff36f2b80327321'},
    {'name': 'networkInterfaceId', 'value': 'eni-0872ab0a1fd81e740'},
    {'name': 'macAddress', 'value': '02:dc:51:b0:0f:89'},
    {'name': 'privateDnsName',
     'value': 'ip-10-0-101-50.ca-central-1.compute.internal'},
    {'name': 'privateIPv4Address', 'value': '10.0.101.50'}]}],
 'attributes': [{'name': 'ecs.cpu-architecture', 'value': 'x86_64'}],
 'availabilityZone': 'ca-central-1a',
 'clusterArn': 'arn:aws:ecs:ca-central-1:142023388927:cluster/quetzal-test',
 'connectivity': 'CONNECTED',
 'connectivityAt': datetime.datetime(2026, 7, 2, 11, 18, 41, 880000, tzinfo=tzlocal()),
 'containers': [{'containerArn': 'arn:aws:ecs:ca-central-1:142023388927:container/quetzal-test/5130d8d0420b4f6c9c169ec079fb7da4/3e1e5a9c-f9fe-489b-8a15-f08f87986b2b',
   'taskArn': 'arn:aws:ecs:ca-central-1:14202

In [ ]:
task_def_arn = task['taskDefinitionArn']
task_def = ecs.describe_task_definition(
	taskDefinition='arn:aws:ecs:ca-central-1:142023388927:task-definition/quetzal-test-task'
)['taskDefinition']
task_def


{'taskDefinitionArn': 'arn:aws:ecs:ca-central-1:142023388927:task-definition/quetzal-test-task:10',
 'containerDefinitions': [{'name': 'quetzal-test',
   'image': '142023388927.dkr.ecr.ca-central-1.amazonaws.com/quetzal-test:12',
   'cpu': 0,
   'portMappings': [],
   'essential': True,
   'environment': [{'name': 'BUCKET_NAME', 'value': 'quetzal-test'},
    {'name': 'IMAGE_TAG', 'value': 'DUMMY'}],
   'mountPoints': [],
   'volumesFrom': [],
   'logConfiguration': {'logDriver': 'awslogs',
    'options': {'awslogs-group': '/aws/ecs/quetzal-test',
     'awslogs-region': 'ca-central-1',
     'awslogs-stream-prefix': 'ecs'}},
   'systemControls': []}],
 'family': 'quetzal-test-task',
 'taskRoleArn': 'arn:aws:iam::142023388927:role/ecs-quetzal-test-task-role',
 'executionRoleArn': 'arn:aws:iam::142023388927:role/ecs-quetzal-test-exec-role',
 'networkMode': 'awsvpc',
 'revision': 10,
 'volumes': [],
 'status': 'ACTIVE',
 'requiresAttributes': [{'name': 'com.amazonaws.ecs.capability.logging-

In [ ]:
task_def_arn = task['taskDefinitionArn']
task_def = ecs.describe_task_definition(taskDefinition=task_def_arn)['taskDefinition']

In [ ]:
for container in task_def['containerDefinitions']:
	print(container['name'])
	print(container['image'])


quetzal-test
142023388927.dkr.ecr.ca-central-1.amazonaws.com/quetzal-test:12


In [ ]:
container = task_def['containerDefinitions'][0]
container['image'].split(':')[1]

'12'

'arn:aws:ecs:ca-central-1:142023388927:task-definition/quetzal-test-task:10'

In [ ]:
task_def = ecs.describe_task_definition(
	taskDefinition='arn:aws:ecs:ca-central-1:142023388927:task-definition/quetzal-test-task'
)['taskDefinition']

In [ ]:
task_def['containerDefinitions'][0]['image']

'142023388927.dkr.ecr.ca-central-1.amazonaws.com/quetzal-test:12'

In [ ]:
test = {
	'steps': "[{'name': 'step 1', 'path': 'notebooks/2_model/test_1.ipynb'}, {'name': 'step 2', 'path': 'notebooks/2_model/test_2.ipynb'}]"
}

In [229]:
steps = test['steps']

In [239]:
steps

"[{'name': 'step 1', 'path': 'notebooks/2_model/test_1.ipynb'}, {'name': 'step 2', 'path': 'notebooks/2_model/test_2.ipynb'}]"

In [240]:
json.dumps('salut')

'"salut"'

In [241]:
json.loads(json.dumps('salut'))

'salut'

In [ ]:
import os

os.path.exists('infra')


True

In [ ]:
from dataclasses import dataclass


@dataclass
class Status:
	step: str
	error: str | None = None

In [254]:
status = Status(step='')

In [256]:
status.__dict__

{'step': '', 'error': None}

In [257]:
asdict(status)

{'step': '', 'error': None}

In [ ]:
from services.cognito_auth.models import Status, StepStatus

Status(job_id='PREPidARING', status='PREPARING', step_status=None)

Status(job_id='PREPidARING', status=<JobStatus.PREPARING: 'PREPARING'>, step_status=None)

In [ ]:
import os
import json
import boto3

REGION = 'ca-central-1'
s3 = boto3.client('s3', region_name=REGION)


def get_step(bucket: str, scenario: str) -> StepStatus:
	try:
		response = s3.get_object(Bucket=bucket, Key=os.path.join(scenario, 'status.json'))
		body = response['Body'].read().decode('utf-8')
		StepStatus(**json.loads(body))
	except:
		return None
	return body

In [ ]:
b = get_step('quetzal-test', 'test_api_2')

In [273]:
json.loads(b)

{'step': {'name': 'step 2', 'path': 'notebooks/2_model/test_2.ipynb'},
 'error': None}

In [ ]:
test = {'step': 'step 2', 'error': None}

In [276]:
Status(job_id='PREPidARING', status='PREPARING', step_status=test)

Status(job_id='PREPidARING', status=<JobStatus.PREPARING: 'PREPARING'>, step_status=StepStatus(step='step 2', error=None))

In [278]:
StepStatus(**test)

StepStatus(step='step 2', error=None)

In [279]:
StepStatus()

StepStatus(step='', error=None)

In [ ]:
local_dir = 'tmp/outputs/'
if os.path.exists(local_dir):
	for root, _, files in os.walk(local_dir):
		for file in files:
			local_path = os.path.join(root, file)
			print(root, local_path)
			s3_path = os.path.join(root.replace('tmp', prefix), file)

tmp/outputs/ tmp/outputs/file2.txt
tmp/outputs/ tmp/outputs/test.txt
tmp/outputs/am tmp/outputs/am/am_file.txt


In [307]:
prefix = 'salut'
root.replace('tmp', prefix)

'salut/outputs/am'

In [ ]:
	s3 = boto3.resource('s3')
	bucket = s3.Bucket(bucket_name)
	if os.path.exists(local_dir):
		for root, _, files in os.walk(local_dir):
			for file in files:
				local_path = os.path.join(root, file)
				folder = ''
				if root != local_dir:  # if not. return '.' and the os.path.join send root files to ./
					folder = os.path.relpath(root, local_dir)
				s3_path = os.path.join(prefix, folder, file)
				bucket.upload_file(local_path, s3_path, ExtraArgs={'Metadata': metadata})